In [ ]:
import polars as pl
from sentence_transformers import SentenceTransformer

In [ ]:
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
model = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
def get_embeddings_similarity(documents: list[str]) -> list[list[float]]:
    # 2. Calculate embeddings by calling model.encode()
    embeddings = model.encode(documents)

    # 3. Calculate the embedding similarities
    similarities = model.similarity(embeddings, embeddings)
    print(similarities)

In [ ]:
db_uri = "sqlite://../../app/ainterviewer.sqlite"
conversations = pl.read_database_uri(
    uri=db_uri, query="SELECT * FROM conversation", engine="adbc"
)
messages = pl.read_database_uri(
    uri=db_uri, query="SELECT * FROM message", engine="adbc"
)

In [ ]:
result_dict = conversations[["id", "interviewer"]].to_dict(as_series=False)
id_interviewer_dict = dict(zip(result_dict["id"], result_dict["interviewer"]))

messages = (
    messages.with_columns(
        created_at=pl.col("created_at").str.to_datetime(
            format="%Y-%m-%d %H:%M:%S%.6f", time_unit="ms"
        )
    )
    # Create a time delta column, which indicates the time since first messsage
    # for each conversation
    .lazy()
    .sort(["created_at"])
    .with_columns(
        pl.col("created_at").min().over("conversation_id").alias("first_created_at")
    )
    .with_columns(
        (pl.col("created_at") - pl.col("first_created_at")).alias("time_delta")
    )
    .drop("first_created_at")
    .collect()
).sort(["conversation_id", "message_id"])

In [ ]:
embeddings = model.encode(messages["content"].to_list())

In [ ]:
interview_messages = messages.filter(conversation_id=3).with_columns(
    embedding=pl.col("content").map_elements(
        lambda s: model.encode([s])[0].tolist(),
        return_dtype=pl.List(pl.Float64),
    ),
)

In [ ]:
all_main_questions = interview_messages.filter(sub_question=0, role="assistant")
all_main_questions_embeddings = all_main_questions["embedding"]

for main_question in (
    interview_messages["main_question"].unique(maintain_order=True).drop_nulls()
):
    main_question_messages = interview_messages.filter(main_question=main_question)
    for answer in main_question_messages.filter(role="user").iter_rows():
        embeddings = all_main_questions_embeddings.to_list() + [answer[-1]]
        sim_matrix = model.similarity(embeddings, embeddings)
        # check if relevant main question is most similar
        main_question_sim = sim_matrix[-1][:-1]
        # Get the index of the most similar main question
        if (
            not (most_sim_main_question := int(main_question_sim.argmax()))
            == main_question
        ):
            print(main_question, answer[2])
            print(main_question_sim)
            print(
                most_sim_main_question,
                all_main_questions.filter(main_question=most_sim_main_question)[
                    "content"
                ][0],
            )
            print()
    print("-" * 80)

In [ ]:
all_main_questions.filter(main_question=most_sim_main_question)["content"][0]

In [ ]:
all_main_questions.filter(main_question=most_sim_main_question)

In [ ]:
int(main_question_sim.argmax())

In [ ]:
model.similarity(all_main_questions["embedding"], all_main_questions["embedding"])

In [ ]:
all_main_questions["content"]

In [ ]:
main_question_messages

In [ ]:
main_question